In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from products import Products
import json

In [4]:
with open('products.json', "r") as f:
    products = Products(json.loads(f.read()))

In [5]:
from mac import build_urls, get_product_prices

In [6]:
# loop through so you get all the possible ids for all of the product variations
product_ids = [variation.id for product in products.root.values() for variation in product if variation.id is not None]
urls = build_urls(product_ids)
responses = await get_product_prices(urls)

In [7]:
from collections import defaultdict
import re

from bs4 import BeautifulSoup


productPrices = defaultdict(dict)
for key, item in responses.items():
    for id, item in item.body.financing.items():
        # this does
        # productPrices[id] = item
        # and also create a new if it does not exist in one line
        # print(item.priceData.fullPrice.raw.price, key)
        if key == "IN":
            # parse using bs4
            soup = BeautifulSoup(item.priceData.fullPrice.raw.price, 'html.parser')
            productPrices[id].update({key: float(soup.find('span', {'class': 'as-price-currentprice'}).text.strip("₹").split(".")[0])})
        else:
            # only replace .00 with nothing if it is at the end of the string
            productPrices[id].update({key: float(re.sub(r'(?<=\d)\.00$', '', item.priceData.fullPrice.raw.price.strip("$").replace(",", "")))})

In [8]:
for key, value in productPrices.items():
    for variation in [variation for product in products.root.values() for variation in product if variation.id is not None]:
        if key == variation.id:
           variation.prices = value

In [9]:
products.root

{'MacBook Air': [Variation(screen_size=13, chip='M2', storage='256GB', memory=16, gpu_count=8, id='MC7X4', prices={'AE': 3999.0, 'AT': 1202.0, 'AU': 1599.0, 'BE': 1199.0, 'BR': 12499.0, 'CA': 1299.0, 'CH': 999.0, 'CL': 1179990.0, 'CN': 7999.0, 'CZ': 29990.0, 'DE': 1199.0, 'DK': 8999.0, 'ES': 1199.0, 'FI': 1259.0, 'FR': 1199.0, 'HK': 7799.0, 'HU': 499990.0, 'IE': 1249.0, 'IN': 99900.0, 'IT': 1249.0, 'JP': 148800.0, 'KR': 1390000.0, 'LU': 1159.36, 'MX': 22999.0, 'MY': 4499.0, 'NL': 1199.0, 'NO': 13990.0, 'NZ': 1799.0, 'PH': 59990.0, 'PL': 4999.0, 'PT': 1249.0, 'SE': 13495.0, 'SG': 1399.0, 'TH': 34900.0, 'TR': 46999.0, 'TW': 32900.0, 'GB': 999.0, 'VN': 24999000.0, 'US': 999.0}, form_factor=None, glass=None, connectivity=None, model=None, case=None, size=None),
  Variation(screen_size=13, chip='M3', storage='256GB', memory=16, gpu_count=8, id='MC8K4', prices={'AE': 4599.0, 'AT': 1302.0, 'AU': 1799.0, 'BE': 1299.0, 'BR': 13999.0, 'CA': 1449.0, 'CH': 1099.0, 'CL': 1329990.0, 'CN': 8999.0, 'C

In [10]:
import json
from bs4 import BeautifulSoup
import re

def extract_iphone_prices(html_content, regex):
    soup = BeautifulSoup(html_content, 'html.parser')

    script_tag = soup.find('script', {'id': 'metrics'})
    
    if not script_tag:
        return {}
      
    json_data = json.loads(script_tag.string)

    products = json_data['data']['products']

    standard, plus = {}, {}

    for product in products:
        name = product['name']
        price = product['price']['fullPrice']

        # Improved regex to capture both "Pro" and "Pro Max" correctly.
        # Removed extra space.
        match = re.search(regex, name)
        if match:
            storage = match.group(1)

            if "Plus" in name or "Max" in name:
                plus[storage] = float(price)
            else:
                standard[storage] = float(price)

    return standard, plus

# import requests
# html_content = requests.get(f"https://www.apple.com/{models["iPhone SE"][1]}").text

# prices = extract_iphone_prices(html_content, models["iPhone SE"][0])
# print(prices)

# for model, storage_prices in prices.items():
#     print(f"iPhone 16 {model}:")

In [11]:
from bs4 import BeautifulSoup
import json
import re

def extract_watch_prices(html_content, model):
    """
    Extracts the prices for different Apple Watch Series 10 variations from the HTML content.

    Args:
        html_content: The HTML content of the Apple Watch Series 10 page.

    Returns:
        A dictionary containing the prices for different watch variations.
        The keys are strings representing the variation (e.g., "46mm Aluminum GPS") and
        the values are floats representing the corresponding prices.
        Returns an empty dictionary if the required data is not found or if an error occurs during parsing.
    """
    try:
        soup = BeautifulSoup(html_content, 'html.parser')

        script_tag = soup.find('script', string=re.compile(r"window\.PRODUCT_SELECTION_BOOTSTRAP"))

        if not script_tag:
            return {}

        json_str = script_tag.string.split(' = ', 1)[1].split("productSelectionData:")[1].split(',"watchProductSelectionDataNoJS"')[0] + "}"
        
        # The json string from the script ends in a semicolon, remove it
        json_data = json.loads(json_str)

        dimension_data = json_data['displayValues']
        prices = {}

        if "Ultra" in model:
            product_key = "watch_cases-49mm"

            if product_key in dimension_data['prices']:
                prices["Apple Watch Ultra 2"] = float(dimension_data['prices'][product_key]['amount'])

            return prices

        
        for case_material_data in dimension_data['watch_cases-dimensionCaseMaterial']['variantOrder']:
          for case_size_data in dimension_data['watch_cases-dimensionCaseSize']['variantOrder']:
            for connection_data in dimension_data['watch_cases-dimensionConnection']['variantOrder']:
                
                case_material = case_material_data
                case_size: str = case_size_data
                connection = connection_data

                key = f"{"Titanium" if "titanium" in case_material.lower() else "Aluminium"}{case_size.rstrip("mm")}{"GPS" if connection.upper() == "GPS" else "GPS + Cellular"}"

                product_key = f"watch_cases-{case_material}-{case_size}-{connection}"

                
                if product_key in dimension_data['prices']:
                  prices[key] = float(dimension_data['prices'][product_key]['amount'])

        return prices

    except (AttributeError, KeyError, json.JSONDecodeError, ValueError) as e:
        print(f"Error extracting watch prices: {e}")

# import requests
# html_content = requests.get(f"https://www.apple.com/{models["Apple Watch Series 10"][1]}").text

# prices = extract_watch_prices(html_content)
# print(prices)

# for model, storage_prices in prices.items():
#     print(f"iPhone 16 {model}:")

In [12]:
from bs4 import BeautifulSoup
import json
import re

def extract_vision_pro_prices(html_content, model):
    """
    Extracts the prices for different Apple Watch Series 10 variations from the HTML content.

    Args:
        html_content: The HTML content of the Apple Watch Series 10 page.

    Returns:
        A dictionary containing the prices for different watch variations.
        The keys are strings representing the variation (e.g., "46mm Aluminum GPS") and
        the values are floats representing the corresponding prices.
        Returns an empty dictionary if the required data is not found or if an error occurs during parsing.
    """
    try:
        soup = BeautifulSoup(html_content, 'html.parser')

        script_tag = soup.find('script', string=re.compile(r"window\.PRODUCT_SELECTION_BOOTSTRAP"))

        if not script_tag:
            return {}

        json_str = script_tag.string.split(' = ', 1)[1].split("productSelectionData:")[1].split(',"products"')[0] + "}"
        
        # The json string from the script ends in a semicolon, remove it
        json_data = json.loads(json_str)

        dimension_data = json_data['displayValues']
        prices = {}
        for storage in ["256gb", "512gb", "1tb"]:
            product_key = f"apple_vision_pro_2024-{storage}"
            if product_key in dimension_data['prices']:
                prices[storage.upper()] = float(dimension_data['prices'][product_key]['amount'])

        return prices

    except (AttributeError, KeyError, json.JSONDecodeError, ValueError) as e:
        print(f"Error extracting watch prices: {e}")

In [13]:
from bs4 import BeautifulSoup
import json
import re

def extract_airpods_prices(html_content, model):
    """
    Extracts the prices for different Apple Watch Series 10 variations from the HTML content.

    Args:
        html_content: The HTML content of the Apple Watch Series 10 page.

    Returns:
        A dictionary containing the prices for different watch variations.
        The keys are strings representing the variation (e.g., "46mm Aluminum GPS") and
        the values are floats representing the corresponding prices.
        Returns an empty dictionary if the required data is not found or if an error occurs during parsing.
    """
    try:
        soup = BeautifulSoup(html_content, 'html.parser')

        script_tag = soup.find('script', string=re.compile(r"window\.PRODUCT_SELECTION_BOOTSTRAP"))

        if not script_tag:
            return {}

        json_str = script_tag.string.split(' = ', 1)[1].split("productSelectionData:")[1].split(',"products"')[0] + "}"
        
        # The json string from the script ends in a semicolon, remove it
        json_data = json.loads(json_str)

        dimension_data = json_data['displayValues']
        prices = {}
        for _, product in dimension_data['prices'].items():
            if product["basePartNumber"] == "MXP63":
                prices["AirPods 4"] = float(product["currentPrice"]['raw_amount'])
            elif product["basePartNumber"] == "MXP93":
                prices["AirPods 4 with Active Noise Cancellation"] = float(product["currentPrice"]['raw_amount'])
            elif product["basePartNumber"] == "MTJV3":
                prices["AirPods Pro 2"] = float(product["currentPrice"]['raw_amount'])
            elif product["basePartNumber"] == "MWW73":
                prices["AirPods Max"] = float(product["currentPrice"]['raw_amount'])
            else:
                print("fff")
        return prices

    except (AttributeError, KeyError, json.JSONDecodeError, ValueError) as e:
        print(f"Error extracting watch prices: {e}")

In [14]:
models = {
    "iPhone 16 Pro": (r"iPhone\s*16\s*Pro(?:\s+Max)?\s+(\d+(?:GB|TB))", "shop/buy-iphone/iphone-16-pro"),
    "iPhone 16": (r"iPhone\s*16(?:\s+Plus)?\s+(\d+(?:GB|TB))", "shop/buy-iphone/iphone-16"),
    "iPhone 16e": (r"iPhone\s*16e(?:\s+Plus)?\s+(\d+(?:GB|TB))", "shop/buy-iphone/iphone-16e"),
    "iPhone 15": (r"iPhone\s*15(?:\s+Plus)?\s+(\d+(?:GB|TB))", "shop/buy-iphone/iphone-15"),
    "Apple Watch Series 10": (r"", "shop/buy-watch/apple-watch"),
    "Apple Watch Ultra 2": (r"", "shop/buy-watch/apple-watch-ultra"),
    "Apple Watch SE": (r"", "shop/buy-watch/apple-watch-se"),
    "Apple Vision Pro": (r"", "shop/buy-vision/apple-vision-pro"),
    "AirPods 4": (r"", "shop/buy-airpods/airpods-4"),
    "AirPods Pro 2": (r"", "shop/buy-airpods/airpods-pro-2"),
    "AirPods Max": (r"", "shop/buy-airpods/airpods-max"),
}

In [15]:
import httpx
import asyncio
import logging
from countries import COUNTRIES

logging.basicConfig(
    level=logging.ERROR,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

async def fetch_price(client, country, url_prefix, model, regex, url_suffix):
    try:
        url = f"https://{url_prefix}{url_suffix}"
        logger.debug(f"Fetching {url} for {country} {model}")
        
        response = await client.get(url, timeout=50.0)
        if response.status_code == 301: # 301 is for redirect for Apple Vision Pro, because it is not available in all countries
            logger.error(f"Timeout fetching {country} {model}")
            return {'error': "HTTP 301", 'country': country, 'model': model}
        
        if "iPhone" in model:
            prices = extract_iphone_prices(response.text, regex)
        elif "Watch" in model:
            prices = extract_watch_prices(response.text, model)
        elif "Vision" in model:
            prices = extract_vision_pro_prices(response.text, model)
        elif "AirPods" in model:
            prices = extract_airpods_prices(response.text, model)
        logger.debug(f"Successfully fetched prices for {country} {model}")
        
        return {
            'country': country,
            'model': model,
            'prices': prices
        }
    except httpx.TimeoutException:
        logger.error(f"Timeout fetching {country} {model}")
        return {'error': 'Request timeout', 'country': country, 'model': model}
    except httpx.HTTPStatusError as e:
        logger.error(f"HTTP error {e.response.status_code} for {country} {model}")
        return {'error': f'HTTP {e.response.status_code}', 'country': country, 'model': model}
    except Exception as e:
        logger.error(f"Unexpected error for {country} {model}: {str(e)}")
        return {'error': str(e), 'country': country, 'model': model}

async def main():
    try:
        limits = httpx.Limits(max_keepalive_connections=5, max_connections=10)
        async with httpx.AsyncClient(limits=limits, timeout=30.0) as client:
            tasks = [
                fetch_price(client, country, url_prefix, model, regex, url_suffix)
                for country, url_prefix in COUNTRIES.items()
                for model, (regex, url_suffix) in models.items()
            ]
            
            logger.info(f"Starting to fetch {len(tasks)} prices")
            results = await asyncio.gather(*tasks)
            
            error_count = 0
            for result in results:
                if 'error' in result:
                    error_count += 1
                    logger.error(f"Error for {result['country']} {result['model']}: {result['error']}")
                    continue
                    
                country = result['country']
                model = result['model']
                prices = result['prices']
                
                for variation in [v for name, product in products.root.items() 
                                for v in product if name == model]:
                    if model == variation.model or variation.model is None or "AirPods" in model:
                        if variation.prices is None:
                            variation.prices = {}
                        if "iPhone" in model :
                            variation.prices.setdefault(country, prices[0][variation.storage])
                        elif "Vision" in model:
                            variation.prices.setdefault(country, prices[variation.storage])
                        elif "Watch" in model:
                            if variation.size is None:
                                key = "Apple Watch Ultra 2"
                            elif variation.case is None:
                                key = "Aluminium"+str(variation.size)+variation.connectivity
                            else:
                                key = variation.case+str(variation.size)+variation.connectivity
                            if key not in prices:
                                logging.error(f"Error: {key} not found in prices from website for {country}")
                                continue
                            variation.prices.setdefault(country, prices[key])
                        elif "AirPods" in model:
                            if variation.model is not None and "4" in variation.model:
                                variation.prices.setdefault(country, prices[variation.model])
                            else:
                                variation.prices.setdefault(country, prices[model])
                    if f"{model} {'Plus' if 'Pro' not in model else 'Max'}" == variation.model:
                        if variation.prices is None:
                            variation.prices = {}
                        variation.prices.setdefault(country, prices[1][variation.storage])
            
            logger.info(f"Completed with {error_count} errors out of {len(tasks)} requests")

    except Exception as e:
        logger.error(f"Fatal error in main: {str(e)}")
        raise

# Execute the async function
await main()

2025-02-20 17:16:58,054 - ERROR - Timeout fetching AT Apple Vision Pro
2025-02-20 17:17:00,764 - ERROR - Timeout fetching BE Apple Vision Pro
2025-02-20 17:17:01,803 - ERROR - Timeout fetching BR Apple Vision Pro
2025-02-20 17:17:03,641 - ERROR - Timeout fetching CH Apple Vision Pro
2025-02-20 17:17:05,286 - ERROR - Timeout fetching CL Apple Vision Pro
2025-02-20 17:17:06,925 - ERROR - Timeout fetching CZ Apple Vision Pro
2025-02-20 17:17:09,570 - ERROR - Timeout fetching DK Apple Vision Pro
2025-02-20 17:17:10,565 - ERROR - Timeout fetching ES Apple Vision Pro
2025-02-20 17:17:11,753 - ERROR - Timeout fetching FI Apple Vision Pro
2025-02-20 17:17:16,015 - ERROR - Timeout fetching HU Apple Vision Pro
2025-02-20 17:17:18,111 - ERROR - Timeout fetching IE Apple Vision Pro
2025-02-20 17:17:19,509 - ERROR - Timeout fetching IN Apple Vision Pro
2025-02-20 17:17:21,143 - ERROR - Timeout fetching IT Apple Vision Pro
2025-02-20 17:17:24,516 - ERROR - Timeout fetching LU Apple Vision Pro
2025-0

In [16]:
# from countries import COUNTRIES
# for country, url_prefix in COUNTRIES.items():
#     for model, (regex, url_suffix) in models.items():
#         html_content = requests.get(f"https://{url_prefix}{url_suffix}").text
#         prices = extract_iphone_prices(html_content, regex)
#         for variation in [variation for name, product in products.root.items() for variation in product if name == model]:
#             if model == variation.model:
#                 if variation.prices is None:
#                     variation.prices = {}
#                 variation.prices.setdefault(country, prices[0][variation.storage])
#             if f"{model} {'Plus' if 'Pro' not in model else 'Max'}" == variation.model:
#                 if variation.prices is None:
#                     variation.prices = {}
#                 variation.prices.setdefault(country, prices[1][variation.storage])

In [17]:
products.root

{'MacBook Air': [Variation(screen_size=13, chip='M2', storage='256GB', memory=16, gpu_count=8, id='MC7X4', prices={'AE': 3999.0, 'AT': 1202.0, 'AU': 1599.0, 'BE': 1199.0, 'BR': 12499.0, 'CA': 1299.0, 'CH': 999.0, 'CL': 1179990.0, 'CN': 7999.0, 'CZ': 29990.0, 'DE': 1199.0, 'DK': 8999.0, 'ES': 1199.0, 'FI': 1259.0, 'FR': 1199.0, 'HK': 7799.0, 'HU': 499990.0, 'IE': 1249.0, 'IN': 99900.0, 'IT': 1249.0, 'JP': 148800.0, 'KR': 1390000.0, 'LU': 1159.36, 'MX': 22999.0, 'MY': 4499.0, 'NL': 1199.0, 'NO': 13990.0, 'NZ': 1799.0, 'PH': 59990.0, 'PL': 4999.0, 'PT': 1249.0, 'SE': 13495.0, 'SG': 1399.0, 'TH': 34900.0, 'TR': 46999.0, 'TW': 32900.0, 'GB': 999.0, 'VN': 24999000.0, 'US': 999.0}, form_factor=None, glass=None, connectivity=None, model=None, case=None, size=None),
  Variation(screen_size=13, chip='M3', storage='256GB', memory=16, gpu_count=8, id='MC8K4', prices={'AE': 4599.0, 'AT': 1302.0, 'AU': 1799.0, 'BE': 1299.0, 'BR': 13999.0, 'CA': 1449.0, 'CH': 1099.0, 'CL': 1329990.0, 'CN': 8999.0, 'C

In [18]:
filtered_products = {}

for product_name, variations in products.root.items():
    # Filter variations with prices
    filtered_variations = [v for v in variations if v.prices is not None]
    
    # Only include product if it has variations with prices
    if filtered_variations:
        filtered_products[product_name] = filtered_variations

# Create new product structure and save
filtered_product_model = type(products)(filtered_products)

with open("filled_products.json", "w") as f:
    f.write(filtered_product_model.model_dump_json())